# Task 09 — Predict function

**Wave 2** · target file: `backend/model.py` · prerequisites: task 02, task 03, task 06

**🎯 Goal:** `predict_z(images, tabular)` — the placeholder fusion.

Part of the *2026-06-17 fused photo-z backend*. Plan & deps: `../PLAN.md` · conventions: `../README.md`.

In [1]:
# === Setup: run me first. Hops to the repo root so `import backend` and models/ resolve,
# and pulls the model artifact(s) from GCS if missing (needs gcloud auth - see README). ===
import os, sys
p = os.path.abspath('.')
while p != '/' and not (os.path.isdir(p+'/backend') and os.path.isdir(p+'/notebooks')):
    p = os.path.dirname(p)
os.chdir(p); sys.path.insert(0, p)
print('repo root:', p)
os.makedirs('models', exist_ok=True)
if not os.path.exists('models/fake_image_model.pkl'):
    os.system('gcloud storage cp gs://macrocosm-lewagon/models/fake_image_model.pkl models/fake_image_model.pkl')
print('models:', [f for f in os.listdir('models') if f.endswith('.pkl')])

repo root: /home/joseb/code/zhhrozhh/Macrocosm
models: ['baseline_stack.pkl', 'fake_image_model.pkl']


## What to build
`predict_z(images, tabular=None)` in `backend/model.py`:
- if **`tabular` is not None** (an `(n,16)` matrix) → `baseline.predict(X)`,
- else → `image.predict(images)`,
- return a **`list[float]`**.

This is the placeholder fusion: image-only uses the (fake) image model; with tabular it uses the real
baseline. The future fused model keeps this same signature. In `backend/model.py` get the models via
`get_models()` (not the locals below).

## 🛠️ Develop & test here first
Fill the `# TODO` so the asserts pass. **Don't** paste the answer — write it.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from backend.fake_model import RandomRedshiftModel
baseline = LinearRegression().fit(np.random.rand(30, 16), np.random.rand(30) * 0.4)
image = RandomRedshiftModel(0)

def predict_z(images, tabular=None):
    if tabular is not None:
        return baseline.predict(tabular).tolist()
    else:
        return image.predict(images).tolist()


z = predict_z(np.random.rand(2, 64, 64, 5).astype("float32"))     # image-only
assert len(z) == 2 and all(isinstance(v, float) for v in z)
zt = predict_z(None, np.random.rand(3, 16))                       # tabular
assert len(zt) == 3
print("OK image-only:", [round(v, 3) for v in z], "| tabular:", [round(v, 3) for v in zt])

OK image-only: [0.23, 0.109] | tabular: [0.284, 0.101, 0.176]


## ➡️ Move it into `backend/model.py`
Once the cell above passes, put your implementation into **`backend/model.py`** (replace the `# TODO`). Then run the **Check** cell — it imports from `backend/` and verifies the real module.

## ✅ Check (imports from `backend/`)

In [3]:
import os, joblib, numpy as np, tempfile, importlib
from sklearn.linear_model import LinearRegression
from backend.fake_model import RandomRedshiftModel
TMP = tempfile.mkdtemp()
joblib.dump({"model": LinearRegression().fit(np.random.rand(30, 16), np.random.rand(30)), "features": list(range(16))}, f"{TMP}/b.pkl")
joblib.dump(RandomRedshiftModel(0), f"{TMP}/i.pkl")
os.environ["BASELINE_PATH"] = f"{TMP}/b.pkl"; os.environ["IMAGE_MODEL_PATH"] = f"{TMP}/i.pkl"
import backend.config, backend.model
importlib.reload(backend.config); importlib.reload(backend.model)
assert len(backend.model.predict_z(np.random.rand(2, 64, 64, 5).astype("float32"))) == 2
assert len(backend.model.predict_z(None, np.random.rand(3, 16))) == 3
print("predict_z OK")

/home/joseb/.pyenv/versions/3.10.6/envs/macrocosm/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/joseb/.pyenv/versions/3.10.6/envs/macrocosm/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestRegressor from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/joseb/.pyenv/versions/3.10.6/envs/macrocosm/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWar

predict_z OK


/home/joseb/.pyenv/versions/3.10.6/envs/macrocosm/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


> ⚠️ **06 + 09 both edit `backend/model.py`.** Keep *both* functions when merging — see `MERGE.md`.

## 🚀 Ship it

In [ ]:
# === Commit & push on YOUR OWN branch (run last; repo root after Setup) ===
!git checkout -B task/09-predict-fn
!git add backend/model.py notebooks/tasks-2026-6-17/09-predict-fn/task.ipynb
!git commit -m "09-predict-fn: Predict function"
!git push -u origin task/09-predict-fn
# then merge back into 2026.6.17 -> see MERGE.md in this folder